In [14]:
# For handling the timeseries
import pandas as pd, os, datetime
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Statistical analysis
from scipy import stats
import math

# Import the loader function
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

In [3]:
df, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="hourly",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.08 sec
Select group: 0.03 sec
--- Starting Dask-Native Process ---


/g/data/ng72/ms5578/ID_HW_BARRA/process_code.py:168: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grp_pd_jittered = grp_pd.groupby(['lat', 'lon'], group_keys=False).apply(


Starting final Dask compute...
Dask compute finished.

--- DASK TIMING REPORT ---
duid_setup: 0.00 seconds
csv_bulk_read: 0.22 seconds
filter_and_clean: 0.01 seconds
type_conversion_and_dropna: 0.03 seconds
date_filter: 0.01 seconds
hw_tseries_filter: 10.08 seconds
aggregate_hourly: 0.03 seconds
jitter: 0.25 seconds
final_merge: 0.03 seconds
compute: 76.28 seconds
--- END REPORT ---

Process group: 90.89 sec

--- DEBUG: Calculated Heatwave Days per Generator ---
DUID
MUSSELR1    334
SAPHWF1     206
WRWF1       201
HALLWF2     187
HALLWF1     187
GUNNING1    178
WOODLWN1    171
LKBONNY2    168
GULLRWF1    164
NBHWF1      162
WATERLWF    161
CLEMGPWF    158
LKBONNY3    157
SNOWTWN1    157
BLUFF1      151
TARALGA1    150
MEWF1       143
BOCORWF1    126
BODWF1      122
STWF1       115
SNOWNTH1    109
OAKLAND1    108
COOPGWF1    103
BALDHWF1    101
HDWF1        98
MACARTH1     98
HDWF3        92
HDWF2        92
SNOWSTH1     89
GRANWF1      84
MERCER01     84
CROOKWF2     76
ARWF1        72


In [4]:
reg_input = df[['DUID','time','TOTALMWh', 'EHF_flag']].copy()
reg_input['hour'] = reg_input['time'].dt.hour
reg_input['date'] = reg_input['time'].dt.date
reg_input['date'] = reg_input['EHF_flag'].astype(int)

In [12]:
# 1️⃣ Combine hour and EHF_flag into a single category with 2-digit hours
reg_input['hour_EHF'] = reg_input['hour'].astype(str).str.zfill(2) + '_' + reg_input['EHF_flag'].astype(str)

# 2️⃣ Create dummy variables
dummies = pd.get_dummies(reg_input['hour_EHF'], prefix='hEHF')

# 3️⃣ Check dummy columns
print(dummies.columns)

# 4️⃣ Drop one dummy as reference (first column is common)
dummies = dummies.drop(dummies.columns[0], axis=1)

# 5️⃣ Ensure all numeric
dummies = dummies.astype(int)

# 6️⃣ Prepare X and y
X = sm.add_constant(dummies)   # optional intercept
y = pd.to_numeric(reg_input['TOTALMWh'], errors='raise')

# 7️⃣ Fit regression
model = sm.OLS(y, X).fit()
print(model.summary())


Index(['hEHF_00_0.0', 'hEHF_00_1.0', 'hEHF_01_0.0', 'hEHF_01_1.0',
       'hEHF_02_0.0', 'hEHF_02_1.0', 'hEHF_03_0.0', 'hEHF_03_1.0',
       'hEHF_04_0.0', 'hEHF_04_1.0', 'hEHF_05_0.0', 'hEHF_05_1.0',
       'hEHF_06_0.0', 'hEHF_06_1.0', 'hEHF_07_0.0', 'hEHF_07_1.0',
       'hEHF_08_0.0', 'hEHF_08_1.0', 'hEHF_09_0.0', 'hEHF_09_1.0',
       'hEHF_10_0.0', 'hEHF_10_1.0', 'hEHF_11_0.0', 'hEHF_11_1.0',
       'hEHF_12_0.0', 'hEHF_12_1.0', 'hEHF_13_0.0', 'hEHF_13_1.0',
       'hEHF_14_0.0', 'hEHF_14_1.0', 'hEHF_15_0.0', 'hEHF_15_1.0',
       'hEHF_16_0.0', 'hEHF_16_1.0', 'hEHF_17_0.0', 'hEHF_17_1.0',
       'hEHF_18_0.0', 'hEHF_18_1.0', 'hEHF_19_0.0', 'hEHF_19_1.0',
       'hEHF_20_0.0', 'hEHF_20_1.0', 'hEHF_21_0.0', 'hEHF_21_1.0',
       'hEHF_22_0.0', 'hEHF_22_1.0', 'hEHF_23_0.0', 'hEHF_23_1.0'],
      dtype='object')
                            OLS Regression Results                            
Dep. Variable:               TOTALMWh   R-squared:                       0.005
Model:         

In [18]:
# Assumes `reg_input` is already present in the notebook as a pandas DataFrame.

# 1) Copy and basic checks
df = reg_input.copy()
required = {'hour', 'EHF_flag', 'TOTALMWh'}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"reg_input is missing required columns: {missing}")

# 2) Normalize types
# Ensure hour is a consistent categorical (zero-padded string like "00", "01", ..., "23")
df['hour'] = df['hour'].astype(str).str.zfill(2)

# Convert EHF_flag to binary 0/1
if df['EHF_flag'].dtype == bool:
    df['EHF_flag'] = df['EHF_flag'].astype(int)
else:
    # Try common mappings and numeric coercion
    df['EHF_flag'] = df['EHF_flag'].replace({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    df['EHF_flag'] = pd.to_numeric(df['EHF_flag'], errors='coerce')
    if df['EHF_flag'].isnull().any():
        raise ValueError("EHF_flag contains values that cannot be converted to binary 0/1")

# Ensure response is numeric
df['TOTALMWh'] = pd.to_numeric(df['TOTALMWh'], errors='raise')

# 3) Quick diagnostics (optional but helpful)
print("Counts per hour:")
print(df['hour'].value_counts().sort_index())
print("\nEHF_flag value counts:")
print(df['EHF_flag'].value_counts())

# 4) Fit the linear model: TOTALMWh ~ hour (categorical) + EHF_flag (binary)
# Using statsmodels formula interface - C(hour) treats hour as categorical
model = smf.mixedlm('TOTALMWh ~ C(hour) * EHF_flag', reg_input, groups=reg_input['DUID']).fit()

# 5) Show results
print("\nModel summary:")
print(model.summary())

# The fitted model object is available as `model` for further inspection (params, predict, etc.)

Counts per hour:
hour
00    157474
01    152378
02    152215
03    152071
04    152003
05    151919
06    151806
07    151267
08    150668
09    150011
10    149281
11    148937
12    149215
13    149645
14    150265
15    150628
16    150811
17    151117
18    151745
19    152138
20    152340
21    152628
22    152654
23    152583
Name: count, dtype: int64

EHF_flag value counts:
EHF_flag
0.0    3509607
1.0     126192
Name: count, dtype: int64

Model summary:
               Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   TOTALMWh      
No. Observations:    3635799   Method:               REML          
No. Groups:          55        Scale:                2288.8646     
Min. group size:     10887     Log-Likelihood:       -19222152.9193
Max. group size:     122448    Converged:            Yes           
Mean group size:     66105.4                                       
-------------------------------------------------------------------
      

In [16]:
reg_input

,DUID,time,TOTALMWh,EHF_flag,hour,date,hour_EHF
0,ARWF1,2016-06-25 00:00:00+00:00,0.0,0.0,0,0,00_0.0
1,ARWF1,2016-06-25 01:00:00+00:00,0.0,0.0,1,0,01_0.0
2,ARWF1,2016-06-25 02:00:00+00:00,0.0,0.0,2,0,02_0.0
3,ARWF1,2016-06-25 03:00:00+00:00,0.0,0.0,3,0,03_0.0
4,ARWF1,2016-06-25 04:00:00+00:00,0.0,0.0,4,0,04_0.0
...,...,...,...,...,...,...,...
4071287,YENDWF1,2024-02-24 20:00:00+00:00,28.718333,0.0,20,0,20_0.0
4071288,YENDWF1,2024-02-24 21:00:00+00:00,22.523333,0.0,21,0,21_0.0
4071289,YENDWF1,2024-02-24 22:00:00+00:00,25.466667,0.0,22,0,22_0.0
4071290,YENDWF1,2024-02-24 23:00:00+00:00,16.76,0.0,23,0,23_0.0
